In [ ]:
# ============================================
# STEP 1 - IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import shap
import joblib

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix
)

plt.close('all')


# ============================================
# STEP 2 - LOAD FILES
# ============================================

xgb_model = joblib.load(
    r"C:\Users\Dell\Desktop\classification project\financial-fraud-detection\models\fraud_detection_model.pkl"
)

scaler = joblib.load(
    r"C:\Users\Dell\Desktop\classification project\financial-fraud-detection\models\scaler.pkl"
)

X_test = joblib.load(
    r"C:\Users\Dell\Desktop\classification project\financial-fraud-detection\data\processed\X_test.pkl"
)

y_test = joblib.load(
    r"C:\Users\Dell\Desktop\classification project\financial-fraud-detection\data\processed\y_test.pkl"
)

X_train_smote = joblib.load(
    r"C:\Users\Dell\Desktop\classification project\financial-fraud-detection\data\processed\X_train_smote.pkl"
)

print("Files loaded successfully")


# ============================================
# STEP 3 - FRAUD PROBABILITIES
# ============================================

y_prob = xgb_model.predict_proba(X_test)[:, 1]

print(y_prob[:10])


# ============================================
# STEP 4 - ROC CURVE
# ============================================

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_prob
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    label=f'ROC Curve (AUC = {roc_auc:.4f})'
)

plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()


# ============================================
# STEP 5 - PRECISION RECALL CURVE
# ============================================

precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_prob
)

avg_precision = average_precision_score(
    y_test,
    y_prob
)

plt.figure(figsize=(8,6))

plt.plot(
    recall,
    precision,
    label=f'Average Precision = {avg_precision:.4f}'
)

plt.xlabel("Recall")

plt.ylabel("Precision")

plt.title("Precision-Recall Curve")

plt.legend()

plt.show()


# ============================================
# STEP 6 - THRESHOLD TUNING
# ============================================

threshold = 0.7

y_pred_custom = (
    y_prob >= threshold
).astype(int)

cm = confusion_matrix(
    y_test,
    y_pred_custom
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title(f'Confusion Matrix at Threshold {threshold}')

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()


# ============================================
# STEP 7 - SHAP EXPLAINABILITY
# ============================================

explainer = shap.TreeExplainer(xgb_model)

sample_data = X_test.sample(
    1000,
    random_state=42
)

shap_values = explainer.shap_values(sample_data)

print("SHAP values generated successfully")


# ============================================
# STEP 8 - SHAP SUMMARY PLOT
# ============================================

shap.summary_plot(
    shap_values,
    sample_data
)


# ============================================
# STEP 9 - FEATURE IMPORTANCE
# ============================================

importance_df = pd.DataFrame({
    'Feature': X_train_smote.columns,
    'Importance': xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

print(importance_df.head(15))


# ============================================
# STEP 10 - FEATURE IMPORTANCE PLOT
# ============================================

plt.figure(figsize=(10,7))

sns.barplot(
    x='Importance',
    y='Feature',
    data=importance_df.head(15)
)

plt.title("Top 15 Feature Importance")

plt.show()


# ============================================
# STEP 11 - SAVE FEATURE IMPORTANCE
# ============================================

importance_df.to_csv(
    r"C:\Users\Dell\Desktop\classification project\financial-fraud-detection\reports\feature_importance.csv",
    index=False
)

print("Feature importance saved")


# ============================================
# STEP 12 - FRAUD PROBABILITY DISTRIBUTION
# ============================================

plt.figure(figsize=(8,5))

sns.histplot(
    y_prob,
    bins=50
)

plt.title("Fraud Probability Distribution")

plt.xlabel("Fraud Probability")

plt.show()


# ============================================
# STEP 13 - FINAL MESSAGE
# ============================================

print("Model evaluation and explainability completed successfully")